# 🎵 ACE-Step 1.5 no Colab — Versão API + Front-end Web / Gradio
1. **Ambiente de execução → Alterar tipo →** escolha **A100** ou **T4** (GPU). O notebook detecta a GPU e escolhe o modelo sozinho.
2. Configure `SITE_URL` na célula 3 com a URL do seu site na Vercel (ex: `https://seu-site.vercel.app`).
3. Rode a **célula 2** (verificar GPU) e a **célula 3** (iniciar servidor).
4. No final aparecerá o link direto `SITE_URL/?api=<tunnel>&token=<token>` pronto para abrir e usar!
5. Ao terminar, clique em ■ nesta célula ou *Ambiente de execução → Desconectar e excluir* para liberar a GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, re, secrets, shutil, subprocess, time, urllib.request

# ===== CONFIGURAÇÃO =====
MODE = "api"        # "api" = sobe a REST API com proxy CORS (para usar com o front-end web) | "gradio" = interface Gradio clássica
LM_MODEL = "auto"   # "auto" = escolhe sozinho pela GPU (A100 → 4B, T4/L4 → 1.7B). Ou force: "acestep-5Hz-lm-4B" / "acestep-5Hz-lm-1.7B"
REPO = "/content/ACE-Step-1.5"
API_PORT = 8001     # porta interna da acestep-api (usada no modo api)
PORT = 7860         # porta do túnel cloudflared (proxy CORS no modo api ou Gradio no modo gradio)
API_TOKEN = ""      # deixe vazio para gerar um token aleatório seguro automaticamente
SITE_URL = ""       # depois do deploy na Vercel, ex.: "https://suno.vercel.app" (gera um link já conectado)
AUTO_SHUTDOWN_ON_STOP = True   # ao clicar em ■ nesta célula, libera a GPU e para de gastar unidades
# ========================

if MODE == "api" and not API_TOKEN:
    API_TOKEN = secrets.token_urlsafe(16)

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["MPLBACKEND"] = "Agg"   # corrige o erro de backend do matplotlib herdado do Colab

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, text=True, cwd=cwd,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode, r.stdout

def tail(path, n=40):
    try:
        return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception as e:
        return f"(sem log: {e})"

# ---- Detecta a GPU e escolhe o modelo ----
_code, _out = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits")
if _code != 0 or not _out.strip():
    raise SystemExit("❌ Nenhuma GPU encontrada. Vá em Ambiente de execução → Alterar tipo de ambiente de execução e escolha T4 ou A100.")
_name, _mem = [x.strip() for x in _out.strip().splitlines()[0].split(",")[:2]]
_vram_gb = int(_mem) / 1024
if LM_MODEL == "auto":
    LM_MODEL = "acestep-5Hz-lm-4B" if _vram_gb >= 32 else "acestep-5Hz-lm-1.7B"
print(f"GPU: {_name} ({_vram_gb:.0f} GB) → modelo de linguagem: {LM_MODEL} | Modo: {MODE}")

print("1/5 Preparando o uv...")
if not shutil.which("uv"):
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
assert shutil.which("uv"), "uv não foi instalado"

print("2/5 Baixando o ACE-Step...")
if not os.path.isdir(REPO):
    code, out = sh(f"git clone https://github.com/ACE-Step/ACE-Step-1.5.git {REPO}")
    assert code == 0, out
os.chdir(REPO)

print("3/5 Instalando dependências (alguns minutos)...")
code, out = sh("uv sync", cwd=REPO)
if code != 0:
    print(out[-3000:])
    raise SystemExit("❌ Falha no uv sync (veja acima)")

if MODE == "api":
    # o proxy CORS precisa de httpx; só instala se ainda não existir (não altera o pyproject)
    if sh("uv run python -c 'import httpx, fastapi, uvicorn'", cwd=REPO)[0] != 0:
        sh("uv pip install httpx fastapi uvicorn", cwd=REPO)

# Código do Proxy FastAPI com CORS irrestrito e autenticação por token
CORS_PROXY_CODE = '''import os, sys
from fastapi import FastAPI, Request, Response
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
import httpx
import uvicorn

TARGET_URL = os.environ.get("TARGET_API_URL", "http://127.0.0.1:8001")
EXPECTED_TOKEN = os.environ.get("API_TOKEN", "")
PROXY_PORT = int(os.environ.get("PROXY_PORT", "7860"))

app = FastAPI(title="ACE-Step Colab CORS Proxy")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

client = httpx.AsyncClient(base_url=TARGET_URL, timeout=600.0)

@app.api_route("/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS", "HEAD", "PATCH"])
async def proxy_handler(request: Request, path: str):
    if request.method == "OPTIONS":
        return Response(status_code=200)

    # Autenticação por token (se configurado)
    if EXPECTED_TOKEN:
        auth_header = request.headers.get("authorization", "")
        token = ""
        if auth_header.lower().startswith("bearer "):
            token = auth_header[7:].strip()
        if not token:
            token = request.query_params.get("token", "")
        if not token:
            token = request.headers.get("x-api-key", "")
        if not token:
            token = request.headers.get("ai_token", "")

        if token != EXPECTED_TOKEN:
            return Response(
                content='{"code": 401, "error": "Invalid or missing API token", "data": null}',
                status_code=401,
                media_type="application/json"
            )

    req_url = f"/{path}"
    if request.url.query:
        req_url += f"?{request.url.query}"

    req_headers = dict(request.headers)
    req_headers.pop("host", None)
    req_headers.pop("content-length", None)

    body = await request.body()

    try:
        rp_req = client.build_request(
            request.method,
            req_url,
            headers=req_headers,
            content=body,
            timeout=600.0
        )
        rp_resp = await client.send(rp_req, stream=True)

        excluded_headers = ["content-encoding", "content-length", "transfer-encoding", "connection"]
        resp_headers = {
            k: v for k, v in rp_resp.headers.items()
            if k.lower() not in excluded_headers
        }

        return StreamingResponse(
            rp_resp.aiter_raw(),
            status_code=rp_resp.status_code,
            headers=resp_headers,
            background=rp_resp.aclose
        )
    except Exception as e:
        return Response(
            content=f'{{"code": 502, "error": "Proxy error: {str(e)}", "data": null}}',
            status_code=502,
            media_type="application/json"
        )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=PROXY_PORT, log_level="warning")
'''

print("4/5 Ligando a IA (na 1ª vez baixa os modelos, pode levar 5-15 min)...")
sh("pkill -f cloudflared; pkill -f cors_proxy; pkill -f api_server; pkill -f acestep_v15_pipeline")

proxy_proc = None
if MODE == "api":
    with open(os.path.join(REPO, ".env"), "w") as f:
        f.write(f"ACESTEP_CONFIG_PATH=acestep-v15-turbo\nACESTEP_LM_MODEL_PATH={LM_MODEL}\nACESTEP_API_PORT={API_PORT}\nACESTEP_API_HOST=127.0.0.1\nLANGUAGE=en\n")

    logf = open("/content/acestep.log", "w")
    api_env = dict(os.environ, ACESTEP_CONFIG_PATH="acestep-v15-turbo", ACESTEP_LM_MODEL_PATH=LM_MODEL,
                   ACESTEP_API_PORT=str(API_PORT), ACESTEP_API_HOST="127.0.0.1")
    proc = subprocess.Popen(
        ["uv", "run", "python", "-m", "acestep.api_server"],
        cwd=REPO, env=api_env, stdout=logf, stderr=subprocess.STDOUT,
    )

    ready = False
    t0 = time.time()
    for i in range(180):                      # até ~15 min
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(f"http://localhost:{API_PORT}/health", timeout=3)
            ready = True
            break
        except Exception:
            time.sleep(5)
            if i % 12 == 11:
                print(f"   ...{int(time.time()-t0)}s | último log: " + (tail("/content/acestep.log", 1).strip()[:120] or "-"))

    if not ready:
        print("\n❌ A API não subiu. Diagnóstico (me mande um print disto):")
        print("processo ainda vivo:", proc.poll() is None)
        print("--- log ---")
        print(tail("/content/acestep.log", 40))
        print("--- .venv/bin ---")
        print(sh("ls .venv/bin | head -30", cwd=REPO)[1])
        raise SystemExit

    print("✅ ACE-Step API no ar! Subindo proxy CORS...")
    with open("/content/cors_proxy.py", "w") as f:
        f.write(CORS_PROXY_CODE)

    proxylogf = open("/content/proxy.log", "w")
    proxy_env = dict(os.environ, TARGET_API_URL=f"http://127.0.0.1:{API_PORT}", API_TOKEN=API_TOKEN, PROXY_PORT=str(PORT))
    proxy_proc = subprocess.Popen(
        ["uv", "run", "python", "/content/cors_proxy.py"],
        cwd=REPO, env=proxy_env, stdout=proxylogf, stderr=subprocess.STDOUT
    )
    time.sleep(3)
else:
    with open(os.path.join(REPO, ".env"), "w") as f:
        f.write(f"ACESTEP_CONFIG_PATH=acestep-v15-turbo\nACESTEP_LM_MODEL_PATH={LM_MODEL}\nPORT={PORT}\nLANGUAGE=en\n")

    logf = open("/content/acestep.log", "w")
    proc = subprocess.Popen(
        ["uv", "run", "python", "-c", "from acestep.acestep_v15_pipeline import main; main()"],
        cwd=REPO, stdout=logf, stderr=subprocess.STDOUT,
    )

    ready = False
    t0 = time.time()
    for i in range(180):                      # até ~15 min
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(f"http://localhost:{PORT}", timeout=3)
            ready = True
            break
        except Exception:
            time.sleep(5)
            if i % 12 == 11:
                print(f"   ...{int(time.time()-t0)}s | último log: " + (tail("/content/acestep.log", 1).strip()[:120] or "-"))

    if not ready:
        print("\n❌ A IA não subiu. Diagnóstico (me mande um print disto):")
        print("processo ainda vivo:", proc.poll() is None)
        print("--- log ---")
        print(tail("/content/acestep.log", 40))
        print("--- .venv/bin ---")
        print(sh("ls .venv/bin | head -30", cwd=REPO)[1])
        raise SystemExit

print("✅ Servidor pronto!")

print("5/5 Criando o link para o navegador...")
cf = "/content/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, 0o755)
sh("pkill -f cloudflared")
tl = open("/content/tunnel.log", "w")
subprocess.Popen([cf, "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                 stdout=tl, stderr=subprocess.STDOUT)

url = None
for _ in range(40):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tail("/content/tunnel.log", 200))
    if m:
        url = m.group(0)
        break
    time.sleep(2)

print("\n" + "=" * 60)
if MODE == "api":
    print("🎵 BACKEND ACE-STEP 1.5 ATIVO COM SUCESSO!" if url else "⚠️ Link não apareceu. Log do túnel:")
    if url:
        print(f"URL da API (Túnel): {url}")
        print(f"Token de Autenticação: {API_TOKEN}")
        if SITE_URL:
            connect_url = f"{SITE_URL.rstrip('/')}/?api={url}&token={API_TOKEN}"
            print(f"\n🚀 ABRIR SITE COM TUDO CONECTADO:\n{connect_url}")
        else:
            print(f"\n💡 Configure SITE_URL no início da célula com o link do seu site Vercel")
            print(f"   Ou abra seu site e informe:\n   API: {url}\n   Token: {API_TOKEN}")
    else:
        print(tail("/content/tunnel.log", 20))
else:
    print("🎵 ABRA ESTE LINK NO NAVEGADOR:" if url else "⚠️ Link não apareceu. Log do túnel:")
    print(url or tail("/content/tunnel.log", 20))
    if url and SITE_URL:
        print("\n🚀 Link do seu site já conectado:\n" + SITE_URL.rstrip("/") + "/?api=" + url)
print("=" * 60)
print("Deixe esta aba do Colab aberta enquanto estiver usando.")

# ---- Mantém a sessão viva até você parar ----
print("\n🟢 Sessão ativa. Esta célula fica rodando DE PROPÓSITO: com ela em execução o Colab não desconecta por inatividade.")
print("   Para DESLIGAR tudo, clique em ■ (parar) nesta célula." + (" A GPU será liberada." if AUTO_SHUTDOWN_ON_STOP else ""))
_start = time.time()
_n = 0
try:
    while True:
        time.sleep(30)
        _n += 1
        if proc.poll() is not None:
            print("⚠️ O servidor do ACE-Step parou. Últimas linhas do log:")
            print(tail("/content/acestep.log", 30))
            break
        if proxy_proc and proxy_proc.poll() is not None:
            print("⚠️ O proxy CORS parou. Últimas linhas do log:")
            print(tail("/content/proxy.log", 30))
            break
        if _n % 30 == 0:   # a cada 15 min
            print(f"   ainda ligado há {int((time.time()-_start)/60)} min")
except KeyboardInterrupt:
    print("\n⏹️ Parado por você. Encerrando...")
    sh("pkill -f cloudflared")
    sh("pkill -f cors_proxy")
    sh("pkill -f api_server")
    sh("pkill -f acestep_v15_pipeline")
    if AUTO_SHUTDOWN_ON_STOP:
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as e:
            print("Não consegui liberar a GPU sozinho, use: Ambiente de execução → Desconectar e excluir.", e)


### Se algo der errado
Rode a célula abaixo e me mande um print:

In [ ]:
!echo '--- acestep.log ---'; tail -n 40 /content/acestep.log
!echo '--- proxy.log ---'; tail -n 30 /content/proxy.log
!echo '--- tunnel.log ---'; tail -n 10 /content/tunnel.log